In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'bernoulli_abc/job_68372_dense_checkpoints'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# DeepRV ABC Results

This notebook visualizes the exact-GP and DeepRV ABC results across training checkpoints. Change `RUN_NAME` in the next cell to inspect another downloaded experiment. Figures are saved as PDF and PNG inside the run's `notebook_figures` directory.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could Not Find Repository Root")


REPO_ROOT = PROJECT_ROOT
RUN_NAME = 'job_68372_dense_checkpoints'
RESULT_DIR = GENERATED / "bernoulli_abc" / RUN_NAME
FIGURE_DIR = RESULT_DIR / "notebook_figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

FIG_WIDTH = 5.906
EXACT_COLOR = "#0072B2"
DEEPRV_COLOR = "#D55E00"
TRUE_COLOR = "#009E73"

plt.style.use("seaborn-v0_8-white")
plt.rcParams.update({"font.family": "Arial", "axes.labelsize": 10, "axes.titlesize": 11, "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9})


def style_axis(axis):
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.tick_params(direction="out")


def save_figure(figure, name):
    figure.savefig(FIGURE_DIR / f"{name}.pdf", bbox_inches="tight")
    figure.savefig(FIGURE_DIR / f"{name}.png", dpi=300, bbox_inches="tight")


print(f"Results: {RESULT_DIR}")
print(f"Figures: {FIGURE_DIR}")

## Load Results

In [ ]:
with (RESULT_DIR / "config.json").open() as file:
    config = json.load(file)
with (RESULT_DIR / "summary.json").open() as file:
    summary = json.load(file)

observed = np.load(RESULT_DIR / "observed_data.npz")
calibration = np.load(RESULT_DIR / "threshold_calibration.npz")
abc = np.load(RESULT_DIR / "abc_results.npz")
steps = sorted(int(step) for step in summary["deeprv_abc"])
threshold = float(summary["threshold"])
true_lengthscale = float(config["true_lengthscale"])

overview = pd.Series({
    "Locations": config["num_locations"],
    "Domain Start": config["domain_start"],
    "Domain Stop": config["domain_stop"],
    "True Lengthscale": true_lengthscale,
    "Threshold Quantile": summary["threshold_quantile"],
    "ABC Threshold": threshold,
    "Exact Posterior Mean": summary["exact_gp_abc"]["mean"],
    "Exact Posterior Median": summary["exact_gp_abc"]["median"],
    "Exact Posterior Standard Deviation": summary["exact_gp_abc"]["standard_deviation"],
    "Exact Acceptance Rate": summary["exact_gp_abc"]["acceptance_rate"],
}, name="Value").to_frame()
display(overview)

## Observed Dataset

In [ ]:
locations = observed["locations"]
latent = observed["latent"]
probability = observed["probability"]
observations = observed["observations"]

figure, axes = plt.subplots(3, 1, figsize=(FIG_WIDTH, 5.2), sharex=True, constrained_layout=True)
axes[0].plot(locations, latent, color=EXACT_COLOR, linewidth=1.0)
axes[0].axhline(0.0, color="0.55", linewidth=0.8, linestyle=":")
axes[0].set_ylabel("Latent Function")
axes[1].plot(locations, probability, color=DEEPRV_COLOR, linewidth=1.0)
axes[1].axhline(0.5, color="0.55", linewidth=0.8, linestyle=":")
axes[1].set_ylabel("Bernoulli Probability")
axes[1].set_ylim(-0.02, 1.02)
axes[2].scatter(locations, observations, color="0.2", s=7, alpha=0.75)
axes[2].set_xlabel("Input Location")
axes[2].set_ylabel("Observed Outcome")
axes[2].set_yticks([0, 1])
for axis in axes:
    style_axis(axis)
save_figure(figure, "observed_dataset")
plt.show()

## Threshold Calibration

In [ ]:
figure, axis = plt.subplots(figsize=(FIG_WIDTH, 3.65), constrained_layout=True)
axis.hist(calibration["distances"], bins=40, density=True, color=EXACT_COLOR, alpha=0.75, label="Calibration Distances")
axis.axvline(threshold, color=DEEPRV_COLOR, linewidth=1.5, linestyle="--", label=rf"ABC Threshold $\epsilon={threshold:.4f}$")
axis.set_xlabel("ABC Distance")
axis.set_ylabel("Density")
axis.legend(frameon=False)
style_axis(axis)
save_figure(figure, "threshold_calibration")
plt.show()

## Posterior Comparison Across Checkpoints

In [ ]:
exact_samples = abc["exact_accepted"]
bins = np.linspace(0.0, 1.0, 51)
posterior_steps = [100, 250, 500, 1_000, 10_000, 100_000]


figure, axes = plt.subplots(2, 3, figsize=(FIG_WIDTH, 5.0), sharex=True, sharey=True, constrained_layout=True)

for axis, step in zip(axes.flat, posterior_steps):
    deep_samples = abc[f"deeprv_step_{step:06d}_accepted"]
    axis.hist(exact_samples, bins=bins, density=True, color=EXACT_COLOR, linewidth=1.5, label="Exact GP ABC", alpha=0.5)
    axis.hist(deep_samples, bins=bins, density=True, color=DEEPRV_COLOR, linewidth=1.5, label="DeepRV ABC", alpha=0.5)
    axis.axvline(true_lengthscale, color=TRUE_COLOR, linewidth=1.2, linestyle="--", label="True Lengthscale")
    axis.set_title(f"{step:,} Training Steps")
    style_axis(axis)

handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="outside upper center", ncol=3, frameon=False, fontsize=8)

figure.supxlabel("Lengthscale", fontsize=11)
figure.supylabel("Density", fontsize=11)
save_figure(figure, "posterior_comparison")
plt.show()